# Vision-Driven Industrial Safety & Quality Inspection Engine
## Notebook 03 - Train Expanded 12-Class Model

This notebook trains Experiment C: the final YOLOv8 expanded detector using the already-prepared 12-class combined dataset.

Expected inputs:

- `combined_dataset`: 12 classes, already uploaded to Google Drive and available locally under `content_runtime`.
- `best.pt`: improved 7-class textile model checkpoint, available in `Backend` locally and on Drive.

This notebook does not rebuild datasets and does not retrain the improved 7-class model.


In [ ]:
%pip install -q ultralytics PyYAML matplotlib seaborn


---
## Step 1 - Verify GPU Runtime


In [ ]:
import shutil
import subprocess
from pathlib import Path

import torch
import ultralytics

print("=== GPU / RUNTIME CHECK ===", flush=True)
assert torch.cuda.is_available(), "CUDA is not available. Select the connected Google Colab T4 GPU kernel before running training."
props = torch.cuda.get_device_properties(0)
print(f"GPU         : {props.name}", flush=True)
print(f"VRAM        : {props.total_memory / 1e9:.1f} GB", flush=True)
print(f"PyTorch     : {torch.__version__}", flush=True)
print(f"Ultralytics : {ultralytics.__version__}", flush=True)
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu", "--format=csv,noheader"],
        text=True,
    ).strip()
    print(f"nvidia-smi  : {smi}", flush=True)
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}", flush=True)

total, used, free = shutil.disk_usage("/")
print(f"Disk free   : {free / 1e9:.1f} GB", flush=True)
assert free > 10e9, "Less than 10 GB free disk space. Clear Colab runtime storage before training."


---
## Step 2 - Locate and Validate Dataset + Checkpoint


In [ ]:
print("=== NOTEBOOK 3 SETUP STARTED ===", flush=True)

import json
import os
import sys
import zipfile
import shutil
from pathlib import Path

import yaml

EXPECTED_NC = 12
EXPECTED_CLASSES = [
    "baekra", "color issues", "contamination", "cut", "gray stitch", "selvet", "stain",
    "chemical hazard", "fire", "no helmet", "smoke", "water leak",
]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Optional manual overrides. Only fill these in if the automatic resolver prints a clear path error.
MANUAL_COMBINED_DATASET = None  # Example: "/content/drive/MyDrive/Hangzhou_Textile_POC/content_runtime/combined_dataset"
MANUAL_IMPROVED_CHECKPOINT = None  # Example: "/content/drive/MyDrive/Hangzhou_Textile_POC/Backend/best.pt"

IN_COLAB = "google.colab" in sys.modules or (Path("/content").exists() and not sys.platform.startswith("win"))
CONTENT = Path("/content") if IN_COLAB else (Path.cwd() / "content_runtime")
RUNS_DIR = CONTENT / "runs"
REPORTS_DIR = CONTENT / "hazard_reports"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local'}", flush=True)
print(f"Working dir : {Path.cwd()}", flush=True)
print(f"Content dir : {CONTENT}", flush=True)

DRIVE_ROOT = None
DRIVE_PROJECT = None
if IN_COLAB:
    print("Mounting Google Drive...", flush=True)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    assert DRIVE_ROOT.exists(), "Drive mount succeeded but /content/drive/MyDrive is not visible."
    DRIVE_PROJECT = DRIVE_ROOT / "Hangzhou_Textile_POC"
    print(f"Drive root  : {DRIVE_ROOT}", flush=True)
    print(f"Drive project candidate: {DRIVE_PROJECT}", flush=True)


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if IN_COLAB:
        candidates.extend([Path("/content"), DRIVE_PROJECT, DRIVE_ROOT])
    for candidate in candidates:
        if candidate and ((candidate / "final_dataset").exists() or (candidate / "Hazard_Expansion").exists()):
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}", flush=True)


def read_yaml(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}
    return data


def normalize_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names)]
    return list(names or [])


def is_valid_combined_dataset(path: Path) -> bool:
    if not path or not (path / "data.yaml").exists():
        return False
    try:
        cfg0 = read_yaml(path / "data.yaml")
    except Exception:
        return False
    names0 = normalize_names(cfg0.get("names", []))
    return int(cfg0.get("nc", len(names0))) == EXPECTED_NC and names0 == EXPECTED_CLASSES


def combined_dataset_candidates():
    bases = [
        CONTENT / "combined_dataset",
        CONTENT / "combined_dataset" / "combined_dataset",
        PROJECT_ROOT / "content_runtime" / "combined_dataset",
        PROJECT_ROOT / "content_runtime" / "combined_dataset" / "combined_dataset",
        PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
        PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "combined_dataset" / "combined_dataset",
        PROJECT_ROOT / "combined_dataset",
        Path.cwd() / "content_runtime" / "combined_dataset",
        Path.cwd() / "combined_dataset",
    ]
    if DRIVE_ROOT:
        bases.extend([
            DRIVE_ROOT / "combined_dataset",
            DRIVE_ROOT / "combined_dataset" / "combined_dataset",
            DRIVE_ROOT / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "content_runtime" / "combined_dataset" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "combined_dataset" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "content_runtime" / "combined_dataset" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "Hazard_Expansion" / "content_runtime" / "combined_dataset" / "combined_dataset",
            DRIVE_ROOT / "Textile Defect Detection" / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "Textile Defect Detection" / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "Textile-Defect-Detection" / "content_runtime" / "combined_dataset",
            DRIVE_ROOT / "Textile-Defect-Detection" / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
        ])
    if MANUAL_COMBINED_DATASET:
        bases.insert(0, Path(MANUAL_COMBINED_DATASET))
    return bases


def resolve_combined_dataset() -> Path:
    checked = []
    print("Checking known combined_dataset folders...", flush=True)
    for candidate in combined_dataset_candidates():
        candidate = candidate.expanduser()
        checked.append(str(candidate))
        print(f"  check: {candidate}", flush=True)
        if is_valid_combined_dataset(candidate):
            return candidate.resolve()

    # Non-recursive zip fallback only in known project locations.
    zip_candidates = []
    for base in [CONTENT, PROJECT_ROOT, Path.cwd(), DRIVE_PROJECT, DRIVE_ROOT]:
        if base and base.exists():
            zip_candidates.extend(base.glob("*combined_dataset*.zip"))
    zip_candidates = sorted(set(zip_candidates), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)
    if zip_candidates:
        zip_path = zip_candidates[0]
        extract_target = CONTENT / "combined_dataset_from_zip"
        print(f"No valid folder found. Extracting known zip: {zip_path}", flush=True)
        if extract_target.exists():
            shutil.rmtree(extract_target)
        extract_target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_target)
        for candidate in [extract_target, extract_target / "combined_dataset"]:
            checked.append(str(candidate))
            if is_valid_combined_dataset(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not resolve the 12-class combined_dataset. Checked:\n  "
        + "\n  ".join(checked)
        + "\n\nExpected data.yaml with nc: 12 and the verified combined class order. "
        + "If your uploaded Google Drive folder is a shared link, add it as a shortcut in MyDrive or set MANUAL_COMBINED_DATASET above."
    )

COMBINED_DATASET = resolve_combined_dataset()
yaml_path = COMBINED_DATASET / "data.yaml"
cfg = read_yaml(yaml_path)
names = normalize_names(cfg.get("names", []))

assert int(cfg.get("nc", len(names))) == EXPECTED_NC, f"Expected nc={EXPECTED_NC}, got {cfg.get('nc')} in {yaml_path}"
assert names == EXPECTED_CLASSES, f"Class order mismatch. Expected {EXPECTED_CLASSES}, found {names}"

# Write an environment-correct data.yaml. This is required because local Windows paths do not work in Colab.
cfg["path"] = str(COMBINED_DATASET)
cfg["train"] = cfg.get("train", "images/train")
cfg["val"] = cfg.get("val", "images/val")
cfg["test"] = cfg.get("test", "images/test")
cfg["nc"] = EXPECTED_NC
cfg["names"] = names
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)


def split_image_dir(split: str) -> Path:
    value = Path(str(cfg[split]))
    return value if value.is_absolute() else COMBINED_DATASET / value


def split_label_dir(image_dir: Path, split: str) -> Path:
    parts = list(image_dir.parts)
    if "images" in parts:
        parts[parts.index("images")] = "labels"
        return Path(*parts)
    return COMBINED_DATASET / "labels" / split

print("\n=== DATASET VALIDATION ===", flush=True)
split_counts = {}
for split in ["train", "val", "test"]:
    image_dir = split_image_dir(split)
    label_dir = split_label_dir(image_dir, split)
    assert image_dir.exists(), f"Missing {split} image folder: {image_dir}"
    assert label_dir.exists(), f"Missing {split} label folder: {label_dir}"
    image_count = sum(1 for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    label_count = sum(1 for p in label_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt")
    assert image_count > 0, f"No images found in {image_dir}"
    assert label_count > 0, f"No labels found in {label_dir}"
    split_counts[split] = {"image_dir": image_dir, "label_dir": label_dir, "images": image_count, "labels": label_count}
    print(f"{split:<5} images: {image_count:>5} | {image_dir}", flush=True)
    print(f"{split:<5} labels: {label_count:>5} | {label_dir}", flush=True)

max_label_id = -1
for split, info in split_counts.items():
    for label_file in info["label_dir"].glob("*.txt"):
        for line_no, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), start=1):
            if not line.strip():
                continue
            parts = line.split()
            assert len(parts) == 5, f"Invalid YOLO row in {label_file}:{line_no}: {line}"
            cls_id = int(float(parts[0]))
            assert 0 <= cls_id < EXPECTED_NC, f"Class id {cls_id} out of range in {label_file}:{line_no}"
            max_label_id = max(max_label_id, cls_id)
assert max_label_id == EXPECTED_NC - 1, f"Expected max class id 11, found {max_label_id}"

print("\nCombined dataset root:", COMBINED_DATASET, flush=True)
print("Dataset YAML        :", yaml_path, flush=True)
print("Classes             :", names, flush=True)


def checkpoint_candidates():
    candidates = [
        PROJECT_ROOT / "Backend" / "best.pt",
        Path.cwd() / "Backend" / "best.pt",
        CONTENT / "best.pt",
        CONTENT / "runs" / "improved_7class" / "weights" / "best.pt",
        PROJECT_ROOT / "runs" / "improved_7class" / "weights" / "best.pt",
    ]
    if DRIVE_ROOT:
        candidates.extend([
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "Backend" / "best.pt",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "Training_Results" / "yolov8s_improved_7class" / "best.pt",
            DRIVE_ROOT / "Hangzhou_Textile_POC" / "runs" / "improved_7class" / "weights" / "best.pt",
            DRIVE_ROOT / "Backend" / "best.pt",
            DRIVE_ROOT / "Training_Results" / "yolov8s_improved_7class" / "best.pt",
            DRIVE_ROOT / "best.pt",
        ])
    if MANUAL_IMPROVED_CHECKPOINT:
        candidates.insert(0, Path(MANUAL_IMPROVED_CHECKPOINT))
    return candidates

IMPROVED_PT = None
print("\nChecking improved 7-class checkpoint candidates...", flush=True)
for candidate in checkpoint_candidates():
    print(f"  check: {candidate}", flush=True)
    if candidate.exists():
        IMPROVED_PT = candidate.resolve()
        break

if IMPROVED_PT is None:
    raise FileNotFoundError(
        "Improved 7-class best.pt was not found. Set MANUAL_IMPROVED_CHECKPOINT above or place best.pt in Backend/ or Drive Training_Results/yolov8s_improved_7class/."
    )

print("\n=== TRAINING INPUTS READY ===", flush=True)
print(f"Combined dataset : {COMBINED_DATASET}", flush=True)
print(f"YAML             : {yaml_path}", flush=True)
print(f"Starting model   : {IMPROVED_PT}", flush=True)
print(f"Checkpoint size  : {IMPROVED_PT.stat().st_size / 1e6:.1f} MB", flush=True)
print("YOLO will adapt the detection head from the 7-class checkpoint to the 12-class dataset.", flush=True)

PHASE1_NAME = "expanded_12class"
PHASE2_NAME = "expanded_12class_phase2"
EVAL_NAME = "expanded_12class_eval"
DRIVE_RESULTS_DIR = DRIVE_PROJECT / "Training_Results" / "Expanded_12-class_model" if DRIVE_PROJECT else None

print("\n=== OUTPUT PATHS ===", flush=True)
print(f"Phase 1 run : {RUNS_DIR / PHASE1_NAME}", flush=True)
print(f"Phase 2 run : {RUNS_DIR / PHASE2_NAME}", flush=True)
print(f"Eval run    : {RUNS_DIR / EVAL_NAME}", flush=True)
print(f"Drive copy  : {DRIVE_RESULTS_DIR if DRIVE_RESULTS_DIR else 'Drive unavailable'}", flush=True)
print("=== SETUP COMPLETE - READY TO TRAIN ===", flush=True)


---
## Step 3 - Phase 1: Frozen Backbone Training


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import time

print("=== EXPERIMENT C - PHASE 1 / EXPANDED 12-CLASS TRAINING ===", flush=True)
print(f"Starting checkpoint : {IMPROVED_PT}", flush=True)
print(f"Dataset YAML        : {yaml_path}", flush=True)
print("Phase 1             : 20 epochs, freeze first 10 layers", flush=True)

assert Path(yaml_path).exists(), f"Dataset YAML missing: {yaml_path}"
assert Path(IMPROVED_PT).exists(), f"Starting checkpoint missing: {IMPROVED_PT}"

model = YOLO(str(IMPROVED_PT))
t0 = time.time()

results_phase1 = model.train(
    data=str(yaml_path),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=0,
    project=str(RUNS_DIR),
    name=PHASE1_NAME,
    exist_ok=True,
    freeze=10,
    optimizer="SGD",
    lr0=0.005,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    amp=True,
    verbose=True,
    plots=True,
)

phase1_elapsed = (time.time() - t0) / 60
PHASE1_BEST = RUNS_DIR / PHASE1_NAME / "weights" / "best.pt"
PHASE1_LAST = RUNS_DIR / PHASE1_NAME / "weights" / "last.pt"
assert PHASE1_BEST.exists(), f"Phase 1 did not produce best.pt: {PHASE1_BEST}"
assert PHASE1_LAST.exists(), f"Phase 1 did not produce last.pt: {PHASE1_LAST}"
print(f"Phase 1 completed in {phase1_elapsed:.1f} minutes", flush=True)
print(f"Phase 1 best: {PHASE1_BEST}", flush=True)
print(f"Phase 1 last: {PHASE1_LAST}", flush=True)


---
## Step 4 - Phase 2: Full Fine-Tuning


In [ ]:
from ultralytics import YOLO
import time

PHASE1_BEST = RUNS_DIR / PHASE1_NAME / "weights" / "best.pt"
print("=== EXPERIMENT C - PHASE 2 / FULL FINE-TUNING ===", flush=True)
print(f"Phase 1 checkpoint: {PHASE1_BEST}", flush=True)
assert PHASE1_BEST.exists(), f"Phase 1 best checkpoint is missing: {PHASE1_BEST}. Run Phase 1 first."

model2 = YOLO(str(PHASE1_BEST))
t1 = time.time()

results_phase2 = model2.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=-1,
    device=0,
    project=str(RUNS_DIR),
    name=PHASE2_NAME,
    exist_ok=True,
    freeze=0,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=0,
    patience=15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    amp=True,
    verbose=True,
    plots=True,
)

phase2_elapsed = (time.time() - t1) / 60
total_elapsed = phase1_elapsed + phase2_elapsed
EXPANDED_BEST_PT = RUNS_DIR / PHASE2_NAME / "weights" / "best.pt"
EXPANDED_LAST_PT = RUNS_DIR / PHASE2_NAME / "weights" / "last.pt"
RESULTS_CSV = RUNS_DIR / PHASE2_NAME / "results.csv"
assert EXPANDED_BEST_PT.exists(), f"Phase 2 did not produce best.pt: {EXPANDED_BEST_PT}"
assert EXPANDED_LAST_PT.exists(), f"Phase 2 did not produce last.pt: {EXPANDED_LAST_PT}"
assert RESULTS_CSV.exists(), f"Phase 2 did not produce results.csv: {RESULTS_CSV}"
print(f"Phase 2 completed in {phase2_elapsed:.1f} minutes", flush=True)
print(f"Total training time: {total_elapsed:.1f} minutes", flush=True)
print(f"Final best checkpoint: {EXPANDED_BEST_PT}", flush=True)
print(f"Final last checkpoint : {EXPANDED_LAST_PT}", flush=True)
print(f"Results CSV           : {RESULTS_CSV}", flush=True)


---
## Step 5 - Validate Expanded Model and Save Metrics


In [ ]:
from ultralytics import YOLO
import json

print("=== EVALUATING EXPANDED 12-CLASS MODEL ===", flush=True)
EXPANDED_BEST_PT = RUNS_DIR / PHASE2_NAME / "weights" / "best.pt"
assert EXPANDED_BEST_PT.exists(), f"Expanded model checkpoint missing: {EXPANDED_BEST_PT}. Run Phase 2 first."

expanded_model = YOLO(str(EXPANDED_BEST_PT))
val_results = expanded_model.val(
    data=str(yaml_path),
    split="val",
    device=0,
    imgsz=640,
    batch=16,
    verbose=True,
    plots=True,
    project=str(RUNS_DIR),
    name=EVAL_NAME,
    exist_ok=True,
)

box = val_results.box
overall_metrics = {
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
}

class_names = expanded_model.names
per_class = {}
if hasattr(box, "ap_class_index"):
    for i, cls_idx in enumerate(box.ap_class_index):
        name = class_names[int(cls_idx)]
        per_class[name] = {
            "precision": float(box.p[i]),
            "recall": float(box.r[i]),
            "ap50": float(box.ap50[i]),
            "ap50_95": float(box.ap[i]),
        }

HISTORICAL_MAP50 = {
    "baekra": 0.839,
    "color issues": 0.512,
    "contamination": 0.990,
    "cut": 0.828,
    "gray stitch": 0.740,
    "selvet": 0.689,
    "stain": 0.919,
}

print("\n=== EXPERIMENT C - EXPANDED MODEL RESULTS ===", flush=True)
print(f"Precision   : {overall_metrics['precision'] * 100:.2f}%", flush=True)
print(f"Recall      : {overall_metrics['recall'] * 100:.2f}%", flush=True)
print(f"mAP@50      : {overall_metrics['mAP50'] * 100:.2f}%", flush=True)
print(f"mAP@50-95   : {overall_metrics['mAP50_95'] * 100:.2f}%", flush=True)

print(f"\n{'Class':<20} {'P%':>7} {'R%':>7} {'AP@50%':>9} {'AP@50-95%':>11} {'Note':>18}", flush=True)
print("-" * 80, flush=True)
for cls_name in EXPECTED_CLASSES:
    if cls_name not in per_class:
        print(f"{cls_name:<20} -- not evaluated (no validation labels?)", flush=True)
        continue
    m = per_class[cls_name]
    if cls_name in HISTORICAL_MAP50:
        delta = m["ap50"] - HISTORICAL_MAP50[cls_name]
        note = f"TEXTILE {delta * 100:+.1f}%"
    else:
        if m['ap50'] >= 0.70:
            note = "HAZARD HIGH PASS"
        elif m['ap50'] >= 0.60:
            note = "HAZARD SUPPORT"
        else:
            note = "HAZARD FAIL"
    print(
        f"{cls_name:<20} {m['precision'] * 100:>7.1f} {m['recall'] * 100:>7.1f} "
        f"{m['ap50'] * 100:>9.1f} {m['ap50_95'] * 100:>11.1f} {note:>18}",
        flush=True,
    )

print("\n" + "=" * 50, flush=True)
print("DEFINITION OF DONE CHECK", flush=True)
print("Criteria: at least 2 of 5 hazard classes achieve >=70% mAP@50, plus at least 1 of the remaining 3 achieves >=60% mAP@50", flush=True)
print("=" * 50, flush=True)
hazard_classes = ["chemical hazard", "fire", "no helmet", "smoke", "water leak"]
dod_results = {}
for cls_name in hazard_classes:
    ap50 = per_class.get(cls_name, {}).get("ap50", 0.0)
    if ap50 >= 0.70:
        status = "HIGH PASS"
    elif ap50 >= 0.60:
        status = "SUPPORT PASS"
    else:
        status = "FAIL"
    dod_results[cls_name] = {"ap50": ap50, "status": status, "high_pass": ap50 >= 0.70, "support_pass": 0.60 <= ap50 < 0.70}
    print(f"  {cls_name:<15}: {ap50 * 100:.1f}% mAP@50  {status}", flush=True)
n_high_pass = sum(1 for v in dod_results.values() if v["high_pass"])
n_support_pass = sum(1 for v in dod_results.values() if v["support_pass"])
remaining_count = len(hazard_classes) - n_high_pass
dod_overall = n_high_pass >= 2 and n_support_pass >= 1
print(f"\n  High-pass classes (>=70%): {n_high_pass}/5", flush=True)
print(f"  Support-pass classes among remaining {remaining_count} (>=60%): {n_support_pass}/{remaining_count}", flush=True)
print(f"  OVERALL DoD: {'PASS' if dod_overall else 'FAIL'}", flush=True)
print("=" * 50, flush=True)

experiment_c = {
    "experiment": "C - Expanded 12-class",
    "checkpoint": str(EXPANDED_BEST_PT),
    "dataset_yaml": str(yaml_path),
    "run_dir": str(RUNS_DIR / PHASE2_NAME),
    "eval_dir": str(RUNS_DIR / EVAL_NAME),
    "overall": overall_metrics,
    "per_class": per_class,
    "dod": dod_results,
    "dod_rule": "2 of 5 hazard classes >=70% mAP@50 and 1 of remaining 3 >=60% mAP@50",
    "dod_high_pass_count": n_high_pass,
    "dod_support_pass_count": n_support_pass,
    "dod_pass": dod_overall,
    "historical_baseline": {
        "precision": 0.7697,
        "recall": 0.7476,
        "mAP50": 0.7882,
        "mAP50_95": 0.4712,
    },
}
results_path = REPORTS_DIR / "expanded_12class_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(experiment_c, f, indent=2)
print(f"\nSaved Experiment C metrics: {results_path}", flush=True)


---
## Step 6 - Safety Alert Post-Processor Demo


In [ ]:
def process_safety_alerts(detections: list, conf_thresh: float = 0.50) -> dict:
    class_map = {
        7: "chemical hazard",
        8: "fire",
        9: "no helmet",
        10: "smoke",
        11: "water leak",
    }
    alerts = []
    for det in detections:
        class_id = det.get("class_id")
        conf = float(det.get("conf", 0.0))
        if conf >= conf_thresh and class_id in class_map:
            hazard = class_map[class_id]
            severity = "CRITICAL" if class_id in (7, 8, 10) else "WARNING"
            alerts.append({"hazard": hazard, "severity": severity, "conf": conf, "box": det.get("box_xyxy")})
    return {"alert_count": len(alerts), "alerts": alerts}

demo_dets = [
    {"class_id": 9, "box_xyxy": [100, 100, 200, 250], "conf": 0.88},
    {"class_id": 8, "box_xyxy": [400, 200, 550, 400], "conf": 0.92},
]
res = process_safety_alerts(demo_dets)
print("=== INDUSTRIAL SAFETY ALERTS DEMO ===", flush=True)
print(f"Total Alerts: {res['alert_count']}", flush=True)
for alert in res["alerts"]:
    print(f"  [{alert['severity']}] Detected: {alert['hazard']} (conf: {alert['conf'] * 100:.1f}%)", flush=True)


---
## Step 7 - Copy Important Artifacts Back to Drive


In [ ]:
import shutil

EXPANDED_BEST_PT = RUNS_DIR / PHASE2_NAME / "weights" / "best.pt"
EXPANDED_LAST_PT = RUNS_DIR / PHASE2_NAME / "weights" / "last.pt"
RESULTS_CSV = RUNS_DIR / PHASE2_NAME / "results.csv"
assert EXPANDED_BEST_PT.exists(), f"best.pt missing: {EXPANDED_BEST_PT}"
assert EXPANDED_LAST_PT.exists(), f"last.pt missing: {EXPANDED_LAST_PT}"
assert RESULTS_CSV.exists(), f"results.csv missing: {RESULTS_CSV}"
assert results_path.exists(), f"Experiment C JSON missing: {results_path}. Run the evaluation cell first."

if DRIVE_RESULTS_DIR:
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(EXPANDED_BEST_PT, DRIVE_RESULTS_DIR / "best.pt")
    shutil.copy2(EXPANDED_LAST_PT, DRIVE_RESULTS_DIR / "last.pt")
    shutil.copy2(RESULTS_CSV, DRIVE_RESULTS_DIR / "results.csv")
    shutil.copy2(yaml_path, DRIVE_RESULTS_DIR / "data.yaml")
    shutil.copy2(results_path, DRIVE_RESULTS_DIR / "expanded_12class_results.json")

    for source_dir in [RUNS_DIR / PHASE2_NAME, RUNS_DIR / EVAL_NAME]:
        if source_dir.exists():
            target_subdir = DRIVE_RESULTS_DIR / source_dir.name
            target_subdir.mkdir(parents=True, exist_ok=True)
            for pattern in ["*.png", "*.jpg", "*.jpeg", "*.csv"]:
                for file_path in source_dir.glob(pattern):
                    shutil.copy2(file_path, target_subdir / file_path.name)

    print(f"Expanded model artifacts saved to Drive: {DRIVE_RESULTS_DIR}", flush=True)
    print("Top-level contents:", sorted(f.name for f in DRIVE_RESULTS_DIR.iterdir()), flush=True)
else:
    print("Drive not mounted. Artifacts remain in this runtime:", flush=True)
    print(f"  best.pt    : {EXPANDED_BEST_PT}", flush=True)
    print(f"  last.pt    : {EXPANDED_LAST_PT}", flush=True)
    print(f"  results.csv: {RESULTS_CSV}", flush=True)
    print(f"  metrics    : {results_path}", flush=True)


---
## End of Notebook 03

Next: run `04_evaluate_and_compare.ipynb` after this notebook finishes and the Drive artifact copy succeeds.
